# Data Explorer

Load the full daily OHLCV universe from DuckDB, inspect coverage, and get oriented.

In [ ]:
from _setup import *

## 1. Load daily panel

In [ ]:
panel = load_daily_bars(start="2017-01-01", end="2026-12-31")
panel = filter_universe(panel, min_adv_usd=500_000, min_history_days=90)

print(f"Rows: {len(panel):,}")
print(f"Date range: {panel['ts'].min()} to {panel['ts'].max()}")
print(f"Symbols: {panel['symbol'].nunique()}")
panel.head()

## 2. Universe membership over time

In [ ]:
universe_wide = panel.pivot(index="ts", columns="symbol", values="in_universe").fillna(False).astype(bool)
n_in_universe = universe_wide.sum(axis=1)

fig, ax = plt.subplots()
ax.fill_between(n_in_universe.index, n_in_universe.values, alpha=0.3, color=TEAL)
ax.plot(n_in_universe.index, n_in_universe.values, color=TEAL, lw=0.8)
ax.set_title("Number of assets in tradeable universe", fontweight="bold")
ax.set_ylabel("# Assets")
plt.show()

## 3. Top 20 assets by ADV

In [ ]:
close_wide = panel.pivot(index="ts", columns="symbol", values="close")
volume_wide = panel.pivot(index="ts", columns="symbol", values="volume")
returns_wide = close_wide.pct_change(fill_method=None)

dollar_vol = (close_wide * volume_wide).mean()
history_days = close_wide.notna().sum()

summary = pd.DataFrame({
    "avg_adv_usd": dollar_vol,
    "history_days": history_days,
    "first_date": close_wide.apply(lambda s: s.dropna().index.min()),
    "last_date": close_wide.apply(lambda s: s.dropna().index.max()),
}).sort_values("avg_adv_usd", ascending=False)

summary["avg_adv_usd"] = summary["avg_adv_usd"].map(lambda x: f"${x:,.0f}")
summary.head(20)

## 4. Price charts — BTC, ETH, SOL (normalized)

In [ ]:
majors = [s for s in ["BTC-USD", "ETH-USD", "SOL-USD"] if s in close_wide.columns]

fig, ax = plt.subplots()
colors_map = {"BTC-USD": NAVY, "ETH-USD": TEAL, "SOL-USD": GOLD}
for sym in majors:
    prices = close_wide[sym].dropna()
    normed = prices / prices.iloc[0]
    ax.plot(normed.index, normed.values, label=sym, color=colors_map.get(sym, GRAY), lw=1.2)

ax.set_yscale("log")
ax.set_title("Major assets — normalized price (log scale)", fontweight="bold")
ax.set_ylabel("Normalized price")
ax.legend(fontsize=9)
plt.show()

## 5. Return distributions

In [ ]:
fig, axes = plt.subplots(1, len(majors), figsize=(5 * len(majors), 4), sharey=True)
if len(majors) == 1:
    axes = [axes]

for ax, sym in zip(axes, majors):
    ret = returns_wide[sym].dropna()
    ax.hist(ret, bins=100, color=colors_map.get(sym, GRAY), alpha=0.7, edgecolor="white", lw=0.3)
    ax.axvline(ret.mean(), color=RED, ls="--", lw=1, label=f"mean={ret.mean():.4f}")
    ax.set_title(sym, fontweight="bold")
    ax.set_xlabel("Daily return")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Frequency")
fig.suptitle("Daily return distributions", fontweight="bold", fontsize=12)
fig.tight_layout()
plt.show()

## 6. Summary statistics

In [ ]:
stats = []
for sym in majors:
    ret = returns_wide[sym].dropna()
    eq = (1 + ret).cumprod()
    m = compute_metrics(eq)
    m["symbol"] = sym
    stats.append(m)

stats_df = pd.DataFrame(stats).set_index("symbol")
stats_df[["cagr", "vol", "sharpe", "sortino", "max_dd", "skewness"]].round(3)